In [1]:
import sys
print(sys.executable)

/Users/gloriadelriomarquez/Documents/Career/DS4B/00_PROYECTOS/Contratacion de fondos/.venv/bin/python3


In [2]:
try:
    import plotly
    print("Plotly importado desde:", plotly.__file__)
except ImportError as e:
    print("ERROR:", e)

Plotly importado desde: /Users/gloriadelriomarquez/Documents/Career/DS4B/00_PROYECTOS/Contratacion de fondos/.venv/lib/python3.11/site-packages/plotly/__init__.py


In [3]:
# Gráfico EDA continuas 
import numpy as np
import pandas as pd
from math import ceil

from scipy.stats import gaussian_kde
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def dataframe_summary(df):
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "n_nulls": df.isna().sum(),
        "pct_nulls": df.isna().mean() * 100,
        "n_unique": df.nunique()
    })

    return summary.reset_index().rename(columns={"index": "column"})

# summary = dataframe_summary(df)
# summary

In [4]:
# cat = df.select_dtypes('o')
# cont = df.select_dtypes('number')

# Frecuencias variables categóricas
def frecuencias_cat(df_cat):
    resultado = df_cat.apply(lambda x: x.value_counts(normalize = True)).T.stack()\
                .to_frame().reset_index()\
                .rename(columns={'level_0':'variable','level_1':'valor',0: "frecuencia"})\
                .sort_values(by = ['variable','frecuencia'])
    resultado['frecuencia'] = resultado['frecuencia']*100
    return(resultado)

In [5]:
# cat_freq = frecuencias_cat(cat)
# cat_freq

In [26]:
import numpy as np
import pandas as pd

def describe_extended(df_cont):
    desc = df_cont.describe().T
    desc['median'] = df_cont.median()
    desc['skew'] = df_cont.skew()
    desc['kurtosis'] = df_cont.kurtosis()

    cols = ['count', 'mean', 'median', 'std',
            'min', '25%', '50%', '75%', 'max',
            'skew', 'kurtosis']
    desc = desc[cols]

    # Colores
    CORINTO = "#bc4749"   # fuerte
    TIERRA  = "#bc6c25"   # naranja tierra

    # ---- Reglas para skew ----
    def color_skew(val):
        if abs(val) >= 2:
            return f"color: {CORINTO}"
        elif abs(val) >= 1:
            return f"color: {TIERRA}"
        else:
            return ""

    # ---- Reglas para kurtosis ----
    def color_kurtosis(val):
        if abs(val) >= 3:
            return f"color: {CORINTO}"
        elif abs(val) >= 1:
            return f"color: {TIERRA}"
        else:
            return ""

    # ---- Reglas por fila: min/max extremos + std > mean ----
    def row_style(row):
        styles = [""] * len(row)
        idx_map = {col: i for i, col in enumerate(row.index)}

        mean = row["mean"]
        std = row["std"]

        upper = mean + 4 * std
        lower = mean - 4 * std

        # max muy extremo
        if row["max"] > upper:
            styles[idx_map["max"]] = f"color: {CORINTO}"

        # min muy extremo
        if row["min"] < lower:
            styles[idx_map["min"]] = f"color: {CORINTO}"

        # std mayor que la media → mucha dispersión
        if (mean > 0) and (std > mean):
            styles[idx_map["std"]] = f"color: {TIERRA}"

        return styles

    styler = (
        desc.style
        .format("{:.2f}")
        .map(color_skew, subset=['skew'])
        .map(color_kurtosis, subset=['kurtosis'])
        .apply(row_style, axis=1)
    )

    return styler

In [23]:
# Describe extendido para vars numéricas
# def describe_extended(df_cont):
#     desc = df_cont.describe().T
#     desc['median'] = df_cont.median()
#     desc['skew'] = df_cont.skew()
#     desc['kurtosis'] = df_cont.kurtosis()

#     # Reordenar columnas
#     cols = ['count', 'mean', 'median', 'std', 'min', '25%', '50%', '75%', 'max', 'skew', 'kurtosis']
#     desc = desc[cols]
#     return (desc)

# Gráficos EDA categóricos 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from math import ceil

def graficos_eda_categoricos_plotly(df_cat, n_cols=2):
    """
    df_cat : DataFrame solo con variables categóricas
    n_cols : número de columnas deseadas en el grid
    """

    n_vars = df_cat.shape[1]
    filas = ceil(n_vars / n_cols)

    # Crear subplots personalizados
    fig = make_subplots(
        rows=filas,
        cols=n_cols,
        subplot_titles=df_cat.columns,
        horizontal_spacing=0.12,
        vertical_spacing=0.15
    )

    fila = 1
    col = 1

    for col_name in df_cat.columns:

        # Calcular proporciones (%)
        counts = df_cat[col_name].value_counts(normalize=True) * 100

        fig.add_trace(
            go.Bar(
                x=counts.values,
                y=counts.index,
                orientation='h',
                text=[f"{v:.1f}%" for v in counts.values],  # mostrar porcentaje en texto
                textposition='auto'
            ),
            row=fila,
            col=col
        )
        
        # Mover a la siguiente posición
        col += 1
        if col > n_cols:
            col = 1
            fila += 1

    fig.update_layout(
        height=340 * filas,
        width=450 * n_cols,
        showlegend=False,
        title=dict(text="<b>Distribución de variables categóricas (EDA) - Proporciones (%)</b>"),
        #template="simple_white"
    )

    fig.update_xaxes(title_text="Porcentaje (%)")
    fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text}</b>"))

    fig.show(renderer='iframe')

# Ejemplo de uso
# df_cat = df.select_dtypes(include="object")
# 3 columnas
# graficos_eda_categoricos_plotly(df_cat, n_cols=3) 

In [5]:

import numpy as np
import pandas as pd
from math import ceil

from scipy.stats import gaussian_kde
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def graficos_eda_continuas_plotly(
    df_cont,
    n_cols=3,
    nbins=40,
    width_per_subplot=420,
    height_per_subplot=300,
    renderer="plotly_mimetype"
):
    """
    Visualiza distribuciones numéricas mediante histogramas
    normalizados a densidad y curvas KDE.

    Compatible con tipos numéricos anulables de pandas.
    """

    num_vars = df_cont.shape[1]

    if num_vars == 0:
        print("No hay variables numéricas para representar.")
        return None

    filas = ceil(num_vars / n_cols)

    # Espaciado compatible con el número de filas
    vspace = min(0.08, 0.8 / max(filas - 1, 1))

    fig = make_subplots(
        rows=filas,
        cols=n_cols,
        subplot_titles=list(df_cont.columns),
        horizontal_spacing=0.06,
        vertical_spacing=vspace
    )

    for i, col_name in enumerate(df_cont.columns):

        row = i // n_cols + 1
        col = i % n_cols + 1

        # Convertir a NumPy float64 y eliminar valores no finitos
        serie = pd.to_numeric(
            df_cont[col_name],
            errors="coerce"
        ).to_numpy(dtype=np.float64, na_value=np.nan)

        serie = serie[np.isfinite(serie)]

        if serie.size == 0:
            continue

        # Histograma
        fig.add_trace(
            go.Histogram(
                x=serie,
                nbinsx=nbins,
                histnorm="probability density",
                opacity=0.55,
                marker=dict(line=dict(width=0.5)),
                showlegend=False
            ),
            row=row,
            col=col
        )

        # KDE: solo para distribuciones no constantes
        if serie.size > 2 and np.unique(serie).size > 1:

            try:
                kde = gaussian_kde(serie)

                x_vals = np.linspace(
                    serie.min(),
                    serie.max(),
                    250
                )

                y_vals = kde(x_vals)

                fig.add_trace(
                    go.Scatter(
                        x=x_vals,
                        y=y_vals,
                        mode="lines",
                        line=dict(width=2),
                        showlegend=False
                    ),
                    row=row,
                    col=col
                )

            except (ValueError, TypeError, np.linalg.LinAlgError):
                pass

    # Configuración general
    fig.update_layout(
        height=height_per_subplot * filas,
        width=width_per_subplot * n_cols,
        title=dict(
            text="<b>Distribución de variables continuas (Histograma + KDE)</b>",
            x=0.5
        ),
        margin=dict(t=100, l=50, r=40, b=50),
        showlegend=False,
        bargap=0.03
    )

    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True)

    fig.for_each_annotation(
        lambda a: a.update(text=f"<b>{a.text}</b>")
    )

    fig.show(renderer=renderer)

    return fig


# Ejemplo de uso
graficos_eda_continuas_plotly(
    num,
    n_cols=3,
    renderer="plotly_mimetype"
)



## Gain chart

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

def cumulative_gain(y_true, y_scores):
    """
    Devuelve:
      - percent: % de clientes ordenados por score (0–1)
      - gains: % acumulado de positivos capturados (0–1)
    """
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    # Ordenar por probabilidad descendente
    sorted_idx = np.argsort(y_scores)[::-1]
    y_true_sorted = y_true[sorted_idx]

    total_positives = y_true.sum()
    if total_positives == 0:
        raise ValueError("No hay positivos en y_true.")

    gains = np.cumsum(y_true_sorted) / total_positives
    percent = np.arange(1, len(y_true_sorted) + 1) / len(y_true_sorted)

    return percent, gains


def plot_gain_chart(y_true, y_scores, title="Gráfico de Ganancias (Gain Chart)"):
    # 1) Calcular curva
    percent, gains = cumulative_gain(y_true, y_scores)

    # 2) DataFrame para Plotly
    df_gain = pd.DataFrame({
        "percent": percent,
        "Modelo": gains,
        "Aleatorio": percent
    })

    # 3) Pasar a formato largo para tener 2 líneas
    df_long = df_gain.melt(
        id_vars="percent",
        value_vars=["Modelo", "Aleatorio"],
        var_name="Curva",
        value_name="gains"
    )

    # 4) Gráfico
    fig = px.line(
        df_long,
        x="percent",
        y="gains",
        color="Curva",
        labels={
            "percent": "Porcentaje de clientes ordenados por score",
            "gains": "Porcentaje acumulado de positivos capturados",
        },
        title=title,
    )

    fig.update_layout(
        template="plotly_white",
        legend_title="",
    )

    fig.show(renderer="iframe")
    return fig

In [ ]:
# # Ejemplo de uso
# y_scores = modelo.best_estimator_.predict_proba(val_x)[:, 1]
# gain_chart = plot_gain_chart(val_y, y_scores)

## Lift chart

In [4]:
import numpy as np
import pandas as pd
import plotly.express as px

def cumulative_gain_curve(y_true, y_scores):
    """
    Cálculo de Curva de Ganancias Acumuladas (para Lift Chart)
    """
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    sorted_idx = np.argsort(y_scores)[::-1]
    y_sorted = y_true[sorted_idx]

    total_pos = y_true.sum()
    gains = np.cumsum(y_sorted) / total_pos
    percent = np.arange(1, len(y_sorted) + 1) / len(y_sorted)

    return percent, gains


def lift_chart(y_true, modelos_scores, incluir_baseline=True):
    """
    Genera un Lift Chart en Plotly Express.

    Parámetros
    ----------
    y_true : array-like
        Etiquetas reales 0/1
    modelos_scores : dict
        Diccionario con formato:
        {
            "Nombre Modelo 1": scores1, 
            "Nombre Modelo 2": scores2,
            ...
        }
    incluir_baseline : bool
        Si True, añade la línea aleatoria (lift = 1)
    """

    df_lift = [ ]

    for nombre, scores in modelos_scores.items():
        p, g = cumulative_gain_curve(y_true, scores)
        lift = g / p

        df_temp = pd.DataFrame({
            "percent": p,
            "lift": lift,
            "Modelo": nombre
        })
        df_lift.append(df_temp)

    # Baseline aleatoria
    if incluir_baseline:
        percent = np.linspace(0.01, 1, 100)
        df_lift.append(pd.DataFrame({
            "percent": percent,
            "lift": np.ones_like(percent),
            "Modelo": "Aleatorio"
        }))

    df_lift = pd.concat(df_lift, ignore_index=True)

    # Gráfico
    fig = px.line(
        df_lift,
        x="percent",
        y="lift",
        color="Modelo",
        title="Lift Chart",
        labels={
            "percent": "% de clientes contactados",
            "lift": "Lift acumulado"
        },
    )

    fig.update_layout(template="plotly_white", height=600, width=900)
    fig.show(renderer="iframe")
    return fig

In [ ]:
# # Ejemplo de uso
# ac_sco = ac.predict_proba(x)[:, 1]
# rl_sco = rl.predict_proba(x)[:, 1]

# lift_chart(
#     y_true=y,
#     modelos_scores={
#         "Árbol": ac_sco,
#         "Regresión Logística": rl_sco
#     }
# )


## ROC chart 

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.metrics import roc_curve, auc

def plot_roc(model, x, y):
    """
    Curva ROC de un clasificador binario.

    Parameters
    ----------
    model : modelo entrenado con predict_proba
    x : variables predictoras
    y : etiquetas reales (0/1)
    """
    y_proba = model.predict_proba(x)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_proba)
    roc_auc = auc(fpr, tpr)

    fig = px.area(
        x=fpr, y=tpr,
        title=f"ROC Curve (AUC = {roc_auc:.3f})",
        labels={"x": "False Positive Rate", "y": "True Positive Rate"},
    )
    fig.add_shape(type="line", line=dict(dash="dash"), x0=0, x1=1, y0=0, y1=1)
    return fig

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report, roc_curve, auc
)

def classification_model_report(model, X_test, y_test, threshold=0.5, model_name=None, verbose=True):
    """
    Generate a performance report for a binary classification model.
    
    Parameters
    ----------
    model : estimator or search object
        Trained model or search object (e.g. GridSearchCV, RandomizedSearchCV).
        If it has a `best_estimator_` attribute, it will be used automatically.
    X_test : array-like or DataFrame
        Test features.
    y_test : array-like
        True labels for the test set.
    threshold : float, default=0.5
        Threshold to convert predicted probabilities into class predictions.
    model_name : str, optional
        Name of the model to display in the report. If None, it will try to infer it.
    verbose : bool, default=True
        If True, prints an executive summary and a classification report.
    
    Returns
    -------
    metrics_df : pd.DataFrame
        Table with main metrics (AUC, Accuracy, Precision, Recall, F1, TN, FP, FN, TP).
    conf_matrix : np.ndarray
        Confusion matrix [[TN, FP],[FN, TP]].
    clf_report : str
        Text classification report (sklearn style).
    """
    
    # Use best_estimator_ if available (e.g. GridSearchCV, RandomizedSearchCV)
    if hasattr(model, "best_estimator_"):
        estimator = model.best_estimator_
    else:
        estimator = model
    
    # Infer model name if not provided
    if model_name is None:
        model_name = estimator.__class__.__name__
    
    # Predict probabilities and hard predictions
    y_proba = estimator.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    # Metrics
    auc_score   = roc_auc_score(y_test, y_proba)
    accuracy    = accuracy_score(y_test, y_pred)
    precision   = precision_score(y_test, y_pred, zero_division=0)
    recall      = recall_score(y_test, y_pred, zero_division=0)
    f1          = f1_score(y_test, y_pred, zero_division=0)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    metrics_df = pd.DataFrame({
        "model":     [model_name],
        "AUC":       [auc_score],
        "Accuracy":  [accuracy],
        "Precision": [precision],
        "Recall":    [recall],
        "F1":        [f1],
        "TN":        [tn],
        "FP":        [fp],
        "FN":        [fn],
        "TP":        [tp]
    })
    
    # Classification report (text)
    clf_report = classification_report(y_test, y_pred, zero_division=0)
    
    if verbose:
        print(f"\n=== MODEL REPORT: {model_name} ===")
        print(f"AUC:       {auc_score:.3f}")
        print(f"Accuracy:  {accuracy:.3f}")
        print(f"Precision: {precision:.3f}")
        print(f"Recall:    {recall:.3f}")
        print(f"F1-score:  {f1:.3f}")
        print(f"\nConfusion matrix (TN, FP, FN, TP):")
        print(np.array([[tn, fp],
                        [fn, tp]]))
        
        print("\nClassification report:")
        print(clf_report)
        
        # Executive summary
        print("\n=== EXECUTIVE SUMMARY ===")
        print(
            f"The final classification model ({model_name}) shows an AUC of {auc_score:.3f}, "
            f"indicating a solid ability to discriminate between positive and negative cases.\n"
            f"Accuracy is {accuracy:.3f}, with a precision of {precision:.3f} and a recall of {recall:.3f}. "
            f"The F1-score of {f1:.3f} reflects the balance between precision and recall.\n"
            f"The confusion matrix suggests a trade-off between correctly identifying positives (TP = {tp}) "
            f"and limiting false positives (FP = {fp}). Depending on the business objective (e.g., minimizing "
            f"missed opportunities vs. limiting over-contact), the decision threshold {threshold:.2f} can be "
            f"adjusted to prioritize either precision or recall."
        )
    
    return metrics_df, np.array([[tn, fp], [fn, tp]]), clf_report

In [ ]:
# %run utils.ipynb  # o import utils si lo pasas a .py

# metrics_df, conf_matrix, clf_report = model_report(
#     model=model,
#     X_test=X_test,
#     y_test=y_test,
#     threshold=0.5,
#     model_name="Final_LeadScoring_Model"
# )

# metrics_df

In [ ]:
# # Si luego quieres desactivar los prints y solo quedarte con los datos:
# metrics_df, conf_matrix, clf_report = model_report(
#     model=model,
#     X_test=X_test,
#     y_test=y_test,
#     threshold=0.5,
#     verbose=False
# )

## Transactions

In [ ]:

import pandas as pd

def integrate_transactions(tx, tx_v2): 
    """
    Inetgra transactions y transactions_v2. 
    Aquí aun no filtramos por ninguna fecha solo unimos dataframes transaccionales.
    """
    # Comprobamos que ambos datasets tengan mismas columnas 
    assert set(tx1.columns) == set(tx2.columns), "Las columnas de los dfs no coinciden."

    # Eliminamos dupliados 
    tx = tx.drop_duplicates()
    tx_v2 = tx_v2.drop_duplicates()
    
    # Concatenamos los históricos filtrados
    txs_all = pd.concat(
        [tx, tx_v2],
        ignore_index=True
    )

    return txs_all

    
def build_transaction_features(transactions, clients, cutoff):
    """
    Construye features históricas por cliente utilizando
    únicamente transacciones anteriores al cutoff.
    """

    # Fecha de corte según período a predecir (prevención data leakage)
    cutoff = pd.Timestamp(cutoff)

    # Convertir a formato datetime las columnas de tipo fecha
    date_cols = transactions.columns[transactions.columns.str.contains('date')]
    
    transactions[date_cols] = transactions[date_cols].apply(
        lambda c: pd.to_datetime(c.astype("Int64").astype(str), format="%Y%m%d", errors= "raise")
    )

    # Filtrar por clientes (archivo train) y por fecha inferior al período a predecir
    tx = transactions.loc[
        transactions["msno"].isin(clients["msno"])
        & (transactions["transaction_date"] < cutoff)
    ].copy()

    # ----------------------------
    # 2. Variables auxiliares
    # ----------------------------
    
    # Operaciones no canceladas
    tx["non_cancelled"] = (tx["is_cancel"] == 0).astype(int)
    
    # Importe asociado a operaciones no canceladas
    tx["non_cancelled_amount"] = np.where(
        tx["is_cancel"] == 0,
        tx["actual_amount_paid"],
        0
    )
    
    # Indicador de operaciones con precio de catálogo cero
    tx["zero_list_price"] = (tx["plan_list_price"] == 0).astype(int)
    
    # Precio normalizado por 30 días.
    # Evitamos divisiones por cero cuando payment_plan_days = 0.
    tx["payment_per_30d"] = np.where(
        tx["payment_plan_days"] > 0,
        tx["actual_amount_paid"] * 30 /
        tx["payment_plan_days"].replace(0, np.nan),
        np.nan
    )

    # ----------------------------
    # 3. Agregaciones por cliente
    # ----------------------------
    
    tx_features = tx.groupby("msno").agg(

    # Actividad transaccional
    n_transaction_records =("transaction_date", "size"),
    n_transaction_days=("transaction_date", "nunique"),
    first_transaction=("transaction_date", "min"),
    last_transaction=("transaction_date", "max"),

    # Cancelaciones
    n_cancellations=("is_cancel", "sum"),
    cancel_rate=("is_cancel", "mean"),
    n_non_cancelled_transactions=("non_cancelled", "sum"),
    
    # Importes registrados
    avg_amount_paid=("actual_amount_paid", "mean"),
    total_amount_paid=("actual_amount_paid", "sum"),
    payment_amount_std=("actual_amount_paid", "std"),
    total_non_cancelled_amount=("non_cancelled_amount", "sum"),

    # Precio normalizado
    avg_payment_per_30d=("payment_per_30d", "mean"),
    
    # Planes contratados
    avg_plan_days=("payment_plan_days", "mean"),
    max_plan_days=("payment_plan_days", "max"),
    n_distinct_plan_days=("payment_plan_days", "nunique"),
    n_payment_methods=("payment_method_id", "nunique"),
    n_zero_list_price_transactions=("zero_list_price", "sum"),

    # Renovación automática
    auto_renew_rate=("is_auto_renew", "mean"),

    # Vencimiento máximo registrado
    max_expire_date=("membership_expire_date", "max")
    )
    
    # ----------------------------
    # 4. Actividad por cliente y día
    # ----------------------------
        
    daily_activity = (
        tx.groupby(["msno", "transaction_date"])
        .agg(
            n_daily_transactions=("is_cancel", "size"),
            n_daily_cancellations=("is_cancel", "sum"),
            n_daily_non_cancelled=("non_cancelled", "sum")
        )
        .reset_index()
    )
    
    # Indicadores diarios
    daily_activity["multi_transaction_day"] = (
        daily_activity["n_daily_transactions"] > 1
    ).astype(int)
    
    daily_activity["multi_non_cancelled_day"] = (
        daily_activity["n_daily_non_cancelled"] >= 2
    ).astype(int)
    
    daily_activity["multi_mixed_cancel_day"] = (
        (daily_activity["n_daily_cancellations"] >= 1) &
        (daily_activity["n_daily_non_cancelled"] >= 1)
    ).astype(int)
    
    # Agregación final por cliente
    daily_features = daily_activity.groupby("msno").agg(
        max_transactions_same_day=("n_daily_transactions", "max"),
        avg_transactions_per_day=("n_daily_transactions", "mean"),
        n_multi_transaction_days=("multi_transaction_day", "sum"),
        has_multi_non_cancelled_day=("multi_non_cancelled_day", "max"),
        has_mixed_cancel_day=("multi_mixed_cancel_day", "max")
    )

    # --------------------------------------------------
    # 5. Variables de la última fecha observada
    # --------------------------------------------------
    
    # Identificamos todas las operaciones del último día de cada cliente
    last_dates = tx.groupby("msno")["transaction_date"].transform("max")
    
    last_day_tx = tx.loc[tx["transaction_date"].eq(last_dates)]
    
    last_day_features = last_day_tx.groupby("msno").agg(
        last_day_n_transactions=("transaction_date", "size"),
        last_day_any_cancel=("is_cancel", "max"),
        last_day_any_auto_renew=("is_auto_renew", "max"),
        last_day_all_auto_renew=("is_auto_renew", "min"),
        last_day_max_expire=("membership_expire_date", "max"),
        last_day_max_payment=("actual_amount_paid", "max")
    )

    # -------------------------------------------
    # 6. Integración de todas las features
    # -------------------------------------------
    
    client_txs = (
        tx_features
        .join(daily_features, how="left")
        .join(last_day_features, how="left")
    )

    # -------------------------------------------
    # 7. Variables temporales
    # -------------------------------------------
    
    # Días desde la última transacción
    client_txs["recency_days"] = (
        cutoff - client_txs["last_transaction"]
    ).dt.days
    
    # Periodo observado entre primera y última transacción
    client_txs["transaction_span_days"] = (
        client_txs["last_transaction"] -
        client_txs["first_transaction"]
    ).dt.days
    
    # Días hasta el vencimiento máximo registrado
    client_txs["days_to_max_expiry"] = (
        client_txs["max_expire_date"] - cutoff
    ).dt.days
    
    # Días hasta el vencimiento máximo del último día observado
    client_txs["days_to_expiry"] = (
        client_txs["last_day_max_expire"] - cutoff
    ).dt.days
    
    
    # Identifica si el cliente ha tenido alguna vez operaciones múltiples en una fecha
    client_txs["has_multi_transaction_day"] = (
        client_txs["max_transactions_same_day"] > 1
    ).astype(int)
    
    # Proporción de días de actividad con múltiples transacciones
    client_txs["multi_transaction_day_rate"] = (
        client_txs["n_multi_transaction_days"] /
        client_txs["n_transaction_days"]
    )
    
    # Cuantifica número de registros adicionales respecto a una operación por día
    client_txs["extra_transaction_records"] = (
        client_txs["n_transaction_records"] -
        client_txs["n_transaction_days"]
    )

    # -------------------------------------------
    # 8. Resultado final
    # -------------------------------------------
    client_txs = client_txs.reset_index()
    
    # Comprobación: una única fila por cliente
    assert client_txs["msno"].is_unique
    # Comprobación: no se incorporan transacciones de marzo
    assert tx["transaction_date"].max() < cutoff
    
    print(f"Tamaño del dataset: {client_txs.shape}")

    return client_txs


## Members 

In [ ]:
def build_member_features(members, cutoff):
    """
    Prepara atributos de los miembros y calcula
    su antigüedad en la fecha de corte.
    """
    cutoff = pd.Timestamp(cutoff)
    members = members.copy()

     
    members["registration_init_time"] = pd.to_datetime(
        members["registration_init_time"].astype("Int64").astype(str),
        format="%Y%m%d",
        errors="coerce"
    )

    # Calcular antigüedad del cliente en días desde la fecha de registro hasta la fecha de corte 
    members["membership_age_days"] = (
        cutoff - members["registration_init_time"]
    ).dt.days

    # Seleccionar clientes con fecha de registro anterior a la fecha de corte (prevención data leakage)
    members = members.loc[
        members["membership_age_days"].isna()
        | (members["membership_age_days"] >= 0)
    ].copy()

    return members

## Build features

In [ ]:

def build_features(clients, members, transactions, cutoff):
    """
    Integra las features de miembros y transacciones.
    Devuelve una fila por cliente.
    """
    member_features = build_member_features(members, cutoff)

    transaction_features = build_transaction_features(
        transactions, clients, cutoff
    )

    df = (
        clients
        .merge(
            member_features,
            on="msno",
            how="left",
            validate="one_to_one"
        )
        .merge(
            transaction_features,
            on="msno",
            how="left",
            validate="one_to_one"
        )
    )

    assert len(df) == len(clients)
    assert df["msno"].is_unique

    return df
